# LeetCode 1415 — The k-th Lexicographical String of All Happy Strings of Length n bằng Python

Link đề: https://leetcode.com/problems/the-k-th-lexicographical-string-of-all-happy-strings-of-length-n/ · Độ khó: **Medium** · Chủ đề: String, Backtracking (thêm: Combinatorics / đếm)

Skeleton LeetCode:

```python
class Solution:
    def getHappyString(self, n: int, k: int) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu **thứ tự từ điển** (lexicographical order) của chuỗi và vì sao duyệt chữ cái theo thứ tự `a < b < c` trong backtracking sinh ra kết quả **đã sắp xếp sẵn**.
2. Viết được **backtracking** sinh mọi chuỗi thoả điều kiện (ký tự kề nhau khác nhau), biết **dừng sớm** khi đã đếm đủ `k`.
3. Đếm tổ hợp: có đúng `3 · 2^(n-1)` happy string độ dài `n`, mỗi "khối" con có kích thước là lũy thừa của 2.
4. Dùng phép đếm để **xây trực tiếp** chuỗi thứ `k` trong `O(n)` mà không phải sinh hết — kỹ thuật "k-th object" (giống bài 60 Permutation Sequence).
5. Tránh các lỗi: lệch chỉ số `k` (1-indexed), đếm sai tổng số chuỗi, quên trường hợp `k` vượt quá số chuỗi.

## 2. Đề bài & ví dụ

Một chuỗi gọi là **happy** (vui vẻ) nếu:
- chỉ gồm các chữ `'a'`, `'b'`, `'c'`;
- hai ký tự **kề nhau luôn khác nhau**: `s[i] != s[i+1]`.

Ví dụ: `"abc"`, `"ac"`, `"b"`, `"abcbabcbcb"` là happy; `"aa"`, `"baa"`, `"ababbc"` không happy.

Cho hai số nguyên `n` và `k`. Xét danh sách **tất cả** happy string có độ dài đúng `n`, sắp xếp theo **thứ tự từ điển**. Trả về chuỗi **thứ `k`** (đếm từ 1) của danh sách, hoặc chuỗi rỗng `""` nếu danh sách có ít hơn `k` phần tử.

**Ví dụ 1:** `n = 1, k = 3` → `"c"` (danh sách `["a","b","c"]`).

**Ví dụ 2:** `n = 1, k = 4` → `""` (chỉ có 3 chuỗi).

**Ví dụ 3:** `n = 3, k = 9` → `"cab"` (12 chuỗi: `aba, abc, aca, acb, bab, bac, bca, bcb, cab, cac, cba, cbc`).

**Ràng buộc:** `1 <= n <= 10`, `1 <= k <= 100`.

In [1]:
import random
import time
from itertools import product
from collections import deque
from typing import List

examples = [((1, 3), "c"), ((1, 4), ""), ((3, 9), "cab")]
for (n, k), want in examples:
    print(f"n={n}, k={k} -> mong đợi {want!r}")

n=1, k=3 -> mong đợi 'c'
n=1, k=4 -> mong đợi ''
n=3, k=9 -> mong đợi 'cab'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi theo thứ tự từ điển trong Python

Python so sánh chuỗi **từng ký tự từ trái sang phải**; ký tự đầu tiên khác nhau quyết định thứ tự. Với các chuỗi **cùng độ dài** (như bài này), đó chính là thứ tự "tra từ điển". `sorted()` trên list chuỗi cho thứ tự từ điển.

In [2]:
print("aba" < "abc", "acb" < "bab", "cab" > "bcb")
print(sorted(["cab", "aba", "bca", "abc"]))
print(ord("a"), ord("b"), ord("c"))   # so sánh dựa trên mã ký tự

True True True
['aba', 'abc', 'bca', 'cab']
97 98 99


### 3.2 `itertools.product` — liệt kê mọi tổ hợp có lặp

`product("abc", repeat=n)` sinh mọi bộ `n` ký tự (có `3^n` bộ) **theo thứ tự từ điển** nếu chuỗi nguồn đã sắp xếp. Dùng `"".join(t)` để ghép tuple thành chuỗi.

In [3]:
all2 = ["".join(t) for t in product("abc", repeat=2)]
print(len(all2), all2)

9 ['aa', 'ab', 'ac', 'ba', 'bb', 'bc', 'ca', 'cb', 'cc']


### 3.3 Đệ quy / backtracking

Backtracking = xây lời giải **từng bước**; ở mỗi bước thử mọi lựa chọn hợp lệ, đi sâu xuống, rồi **quay lui** (bỏ lựa chọn) để thử lựa chọn kế. Khung chung:

```python
def dfs(path):
    if path đủ dài: ghi nhận; return
    for lựa chọn in các lựa chọn hợp lệ:
        path.append(lựa chọn); dfs(path); path.pop()
```

In [4]:
def all_binary(n):
    out, path = [], []
    def dfs():
        if len(path) == n:
            out.append("".join(path))
            return
        for ch in "01":
            path.append(ch)
            dfs()
            path.pop()          # quay lui
    dfs()
    return out

print(all_binary(3))

['000', '001', '010', '011', '100', '101', '110', '111']


### 3.4 Lũy thừa 2, phép chia nguyên `//`, chia dư `%`, `divmod`

Ở phần xây trực tiếp ta liên tục chia chỉ số cho kích thước khối: `q, r = divmod(x, size)` cho thương `q` (chọn khối nào) và dư `r` (vị trí trong khối). `1 << m` bằng `2**m`.

In [5]:
print(1 << 3, 2 ** 3)
print(divmod(19, 8), 19 // 8, 19 % 8)
print(bin(5), format(5, "03b"))   # biểu diễn nhị phân có đệm 0

8 8
(2, 3) 2 3
0b101 101


### 3.5 `deque` — hàng đợi hai đầu

`collections.deque` cho `append` / `popleft` trong `O(1)`; dùng cho cách sinh theo BFS (mở rộng từng tầng độ dài).

In [6]:
q = deque(["a", "b", "c"])
q.append("ab")
print(q.popleft(), list(q))

a ['b', 'c', 'ab']


## 4. Ý tưởng nền tảng / trực giác

Hãy liệt kê **bằng tay** với `n = 2`:
- Chữ đầu `a` → chữ thứ hai phải khác `a`: `ab`, `ac`.
- Chữ đầu `b` → `ba`, `bc`.
- Chữ đầu `c` → `ca`, `cb`.

Tổng cộng `6 = 3 · 2`. Với `n = 3`, mỗi chuỗi độ dài 2 lại có **đúng 2** cách nối thêm ⇒ `12 = 3 · 2 · 2`. Tổng quát: **chữ đầu có 3 lựa chọn, mỗi chữ sau có 2 lựa chọn** ⇒ `3 · 2^(n-1)` chuỗi.

Vẽ thành **cây**: gốc rỗng có 3 con `a, b, c`; mỗi nút khác có 2 con. Đọc lá từ trái sang phải (khi các con xếp theo `a<b<c`) chính là danh sách đã sắp xếp. Với `n = 3, k = 9`:
- Mỗi chữ đầu "sở hữu" `2^2 = 4` lá. `k = 9` ⇒ bỏ qua khối `a` (lá 1–4), khối `b` (lá 5–8), rơi vào khối `c` ở vị trí 1.
- Trong khối `c`, mỗi lựa chọn chữ thứ hai (`a` hoặc `b`) sở hữu 2 lá; vị trí 1 ⇒ chọn `a`, vị trí 1 trong khối `ca`.
- Chữ thứ ba khác `a`: `b` hoặc `c`, vị trí 1 ⇒ `b`. Kết quả `"cab"` ✔.

In [7]:
# Kiểm chứng trực giác: in cây n = 3 theo từng khối chữ đầu
def happy_list(n):
    return sorted("".join(t) for t in product("abc", repeat=n)
                  if all(t[i] != t[i + 1] for i in range(n - 1)))

lst = happy_list(3)
for i in range(0, len(lst), 4):
    print(f"khối '{lst[i][0]}':", [(i + j + 1, s) for j, s in enumerate(lst[i:i + 4])])

khối 'a': [(1, 'aba'), (2, 'abc'), (3, 'aca'), (4, 'acb')]
khối 'b': [(5, 'bab'), (6, 'bac'), (7, 'bca'), (8, 'bcb')]
khối 'c': [(9, 'cab'), (10, 'cac'), (11, 'cba'), (12, 'cbc')]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** sinh **mọi** chuỗi độ dài `n` trên bảng chữ `abc` (có `3^n` chuỗi), lọc những chuỗi happy, sắp xếp, lấy phần tử `k-1` (đổi sang chỉ số 0) nếu tồn tại.

Đây là cách "thẳng như đề bài", dễ đúng, làm chuẩn để kiểm chứng các cách khác.

In [8]:
def is_happy(s):
    return all(s[i] != s[i + 1] for i in range(len(s) - 1))

def happy_brute(n, k):
    cands = ["".join(t) for t in product("abc", repeat=n)]
    happy = sorted(s for s in cands if is_happy(s))
    return happy[k - 1] if k <= len(happy) else ""

for (n, k), want in examples:
    got = happy_brute(n, k)
    print(f"n={n}, k={k}: {got!r}", "✔" if got == want else "✘")

n=1, k=3: 'c' ✔
n=1, k=4: '' ✔
n=3, k=9: 'cab' ✔


**Độ phức tạp:** sinh `3^n` chuỗi, mỗi chuỗi kiểm tra `O(n)` ⇒ `O(3^n · n)` thời gian, `O(3^n · n)` bộ nhớ nếu giữ cả list. Với `n = 10`: `59049` chuỗi — vẫn chạy được, nhưng phần lớn bị loại (chỉ `1536` là happy).

**Nút thắt:**
1. Sinh cả những chuỗi chắc chắn sai (có `"aa"`) rồi mới loại.
2. Sắp xếp là thừa, vì `product` vốn đã sinh theo thứ tự từ điển.
3. Ta chỉ cần **một** chuỗi thứ `k` (`k ≤ 100`) mà lại sinh hết.

In [9]:
for n in [5, 8, 10]:
    t = time.perf_counter()
    happy_brute(n, 100)
    print(f"n={n}: 3^n={3**n:6d}, happy={3 * 2**(n-1):5d}, thời gian {time.perf_counter() - t:.4f}s")

n=5: 3^n=   243, happy=   48, thời gian 0.0001s
n=8: 3^n=  6561, happy=  384, thời gian 0.0027s
n=10: 3^n= 59049, happy= 1536, thời gian 0.0273s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — cắt tỉa ngay khi sinh:** khi xây chuỗi từng ký tự, chỉ thử những chữ **khác ký tự cuối**. Như vậy ta chỉ sinh happy string, không có chuỗi rác.

**Insight 2 — thứ tự có sẵn:** nếu ở mọi bước ta thử chữ theo thứ tự `a, b, c`, thì các chuỗi hoàn chỉnh xuất hiện **đúng theo thứ tự từ điển** (duyệt cây theo thứ tự trước, con trái trước). ⇒ không cần sort, và có thể **dừng khi đếm đến `k`**.

**Insight 3 — đếm để nhảy cóc:** sau khi đã chọn `i` ký tự đầu, số cách hoàn thành phần còn lại là `2^(n-1-i)` (mỗi vị trí sau còn đúng 2 lựa chọn), **không phụ thuộc** ta đã chọn gì. Do đó:

- Nếu `k > 3 · 2^(n-1)` thì trả `""`.
- Đổi sang chỉ số 0: `x = k - 1`. Chữ đầu = `"abc"[x // 2^(n-1)]`, `x %= 2^(n-1)`.
- Với mỗi vị trí tiếp theo: hai lựa chọn hợp lệ (đã sắp tăng) là `opts`; khối mỗi lựa chọn có kích thước `2^(còn lại)`; **nếu** `x < size` **thì** chọn `opts[0]`, **ngược lại** chọn `opts[1]` và `x -= size`.

**Insight 4 — nhìn dưới dạng nhị phân:** phần `x % 2^(n-1)` viết ở hệ 2 với `n-1` bit; bit `0` = "chọn chữ nhỏ hơn trong 2 chữ còn lại", bit `1` = "chọn chữ lớn hơn". Đẹp và gọn!

## 7. Các cách tiếp theo

### Cách 2 — Backtracking sinh theo thứ tự, dừng sớm ở chuỗi thứ `k`

**Từng đoạn:**
1. `path` là list ký tự đang xây; `count` đếm số chuỗi hoàn chỉnh đã gặp.
2. Khi `len(path) == n`: tăng `count`; nếu `count == k` thì lưu kết quả và báo "xong".
3. Ở mỗi bước duyệt `ch in "abc"`, bỏ qua nếu `path` khác rỗng và `ch == path[-1]`.
4. Hàm `dfs` trả `True` khi đã tìm được — mọi tầng gọi phía trên lập tức `return True` (dừng sớm).

In [10]:
def happy_backtrack(n, k):
    path, count, ans = [], 0, ""

    def dfs():
        nonlocal count, ans
        if len(path) == n:
            count += 1
            if count == k:
                ans = "".join(path)
                return True
            return False
        for ch in "abc":
            if path and path[-1] == ch:
                continue
            path.append(ch)
            if dfs():
                return True
            path.pop()
        return False

    dfs()
    return ans

for (n, k), want in examples:
    print(n, k, repr(happy_backtrack(n, k)), happy_backtrack(n, k) == want)

1 3 'c' True
1 4 '' True
3 9 'cab' True


Code nộp LeetCode (cách 2):

In [11]:
class SolutionBacktrack:
    def getHappyString(self, n: int, k: int) -> str:
        path, self.count, self.ans = [], 0, ""

        def dfs() -> bool:
            if len(path) == n:
                self.count += 1
                if self.count == k:
                    self.ans = "".join(path)
                    return True
                return False
            for ch in "abc":
                if path and path[-1] == ch:
                    continue
                path.append(ch)
                if dfs():
                    return True
                path.pop()
            return False

        dfs()
        return self.ans

print(SolutionBacktrack().getHappyString(3, 9))

cab


**Độ phức tạp:** chỉ thăm các nút trên đường tới lá thứ `k` và các nhánh bên trái nó: `O(k · n)` thời gian (mỗi lá tốn tối đa `n` bước), độ sâu đệ quy `O(n)`. Trường hợp `k` lớn hơn tổng số chuỗi thì phải duyệt hết `O(3 · 2^(n-1) · n)`; với `n ≤ 10` chỉ ~15 nghìn bước.

### Cách 3 — Sinh theo BFS (từng tầng độ dài) bằng hàng đợi

**Ý tưởng:** bắt đầu hàng đợi `["a", "b", "c"]`. Lặp: lấy chuỗi đầu hàng đợi, nếu độ dài `< n` thì nối thêm từng chữ khác chữ cuối (theo thứ tự `a, b, c`) rồi đẩy vào cuối. Vì tầng trước đã có thứ tự từ điển và ta mở rộng theo thứ tự, tầng sau cũng có thứ tự từ điển. Khi phần tử đầu hàng đợi có độ dài `n`, toàn bộ hàng đợi là danh sách đáp án đã sắp.

In [12]:
def happy_bfs(n, k):
    q = deque(["a", "b", "c"])
    while len(q[0]) < n:
        s = q.popleft()
        for ch in "abc":
            if ch != s[-1]:
                q.append(s + ch)
    return q[k - 1] if k <= len(q) else ""

for (n, k), want in examples:
    print(n, k, repr(happy_bfs(n, k)), happy_bfs(n, k) == want)

1 3 'c' True
1 4 '' True
3 9 'cab' True


In [13]:
class SolutionBFS:
    def getHappyString(self, n: int, k: int) -> str:
        q = deque(["a", "b", "c"])
        while len(q[0]) < n:
            s = q.popleft()
            for ch in "abc":
                if ch != s[-1]:
                    q.append(s + ch)
        return q[k - 1] if k <= len(q) else ""

print(SolutionBFS().getHappyString(3, 9))

cab


**Độ phức tạp:** sinh đủ mọi happy string mọi độ dài `≤ n`: khoảng `3 · 2^n` chuỗi, mỗi lần nối chuỗi `O(n)` ⇒ `O(2^n · n)` thời gian và bộ nhớ. Không dừng sớm được như cách 2, nhưng không cần đệ quy.

### Cách 4 — Xây trực tiếp bằng phép đếm (tối ưu)

**Từng đoạn:**
1. `total = 3 * 2^(n-1)`; nếu `k > total` trả `""`.
2. `x = k - 1` (chỉ số 0), `block = 2^(n-1)` = số chuỗi bắt đầu bằng mỗi chữ.
3. Chữ đầu: `"abc"[x // block]`, cập nhật `x %= block`.
4. Lặp `n - 1` lần: `block //= 2`; `opts` = 2 chữ khác chữ cuối (theo thứ tự tăng); `q, x = divmod(x, block)`; nối `opts[q]`.

In [14]:
def happy_direct(n, k):
    block = 1 << (n - 1)          # số chuỗi cho mỗi chữ đầu
    if k > 3 * block:
        return ""
    x = k - 1
    q, x = divmod(x, block)
    res = ["abc"[q]]
    for _ in range(n - 1):
        block //= 2
        opts = [ch for ch in "abc" if ch != res[-1]]   # đã tăng dần
        q, x = divmod(x, block)
        res.append(opts[q])
    return "".join(res)

for (n, k), want in examples:
    print(n, k, repr(happy_direct(n, k)), happy_direct(n, k) == want)

1 3 'c' True
1 4 '' True
3 9 'cab' True


Biến thể dùng **bit** (Insight 4): bit thứ `j` (từ trái) của `x` sau khi lấy chữ đầu cho biết chọn chữ nhỏ (0) hay lớn (1).

In [15]:
def happy_bits(n, k):
    half = 1 << (n - 1)
    if k > 3 * half:
        return ""
    first, rest = divmod(k - 1, half)
    res = ["abc"[first]]
    bits = format(rest, f"0{n - 1}b") if n > 1 else ""
    for b in bits:
        opts = [ch for ch in "abc" if ch != res[-1]]
        res.append(opts[int(b)])
    return "".join(res)

print([happy_bits(n, k) for (n, k), _ in examples])
print("n=3,k=9: rest =", (9 - 1) % 4, "->", format((9 - 1) % 4, "02b"), "-> 'c' rồi chọn nhỏ (a), chọn nhỏ (b) = 'cab'")

['c', '', 'cab']
n=3,k=9: rest = 0 -> 00 -> 'c' rồi chọn nhỏ (a), chọn nhỏ (b) = 'cab'


In [16]:
class SolutionDirect:
    def getHappyString(self, n: int, k: int) -> str:
        block = 1 << (n - 1)
        if k > 3 * block:
            return ""
        q, x = divmod(k - 1, block)
        res = ["abc"[q]]
        for _ in range(n - 1):
            block //= 2
            opts = [ch for ch in "abc" if ch != res[-1]]
            q, x = divmod(x, block)
            res.append(opts[q])
        return "".join(res)

print(SolutionDirect().getHappyString(3, 9), SolutionDirect().getHappyString(10, 100))

cab abacbabacb


**Độ phức tạp:** `n` vòng lặp, mỗi vòng `O(1)` (bảng chữ cỡ 3) ⇒ `O(n)` thời gian, `O(n)` bộ nhớ cho kết quả. Không phụ thuộc `k`!

## 8. TRACE từng bước

Trace cách 4 với `n = 3, k = 9` và `n = 4, k = 20`:

| Bước | `block` | `x` trước | lựa chọn `opts` | `q = x // block` | chữ được chọn | `x` sau |
|---|---|---|---|---|---|---|
| chữ đầu | 4 | 8 | `a b c` | 2 | `c` | 0 |
| vị trí 2 | 2 | 0 | `a b` | 0 | `a` | 0 |
| vị trí 3 | 1 | 0 | `b c` | 0 | `b` | 0 |

⇒ `"cab"`.

In [17]:
def trace_direct(n, k):
    block = 1 << (n - 1)
    print(f"n={n}, k={k}, tổng số happy = {3 * block}")
    if k > 3 * block:
        print("k vượt quá -> ''")
        return ""
    x = k - 1
    q, nx = divmod(x, block)
    res = ["abc"[q]]
    print(f"  chữ đầu : block={block:3d}, x={x:3d}, opts=abc, q={q} -> '{res[-1]}', x mới={nx}")
    x = nx
    for pos in range(2, n + 1):
        block //= 2
        opts = "".join(ch for ch in "abc" if ch != res[-1])
        q, nx = divmod(x, block)
        res.append(opts[q])
        print(f"  vị trí {pos}: block={block:3d}, x={x:3d}, opts={opts}, q={q} -> '{res[-1]}', x mới={nx}")
        x = nx
    print("  kết quả:", "".join(res))
    return "".join(res)

trace_direct(3, 9)
trace_direct(4, 20)

n=3, k=9, tổng số happy = 12
  chữ đầu : block=  4, x=  8, opts=abc, q=2 -> 'c', x mới=0
  vị trí 2: block=  2, x=  0, opts=ab, q=0 -> 'a', x mới=0
  vị trí 3: block=  1, x=  0, opts=bc, q=0 -> 'b', x mới=0
  kết quả: cab
n=4, k=20, tổng số happy = 24
  chữ đầu : block=  8, x= 19, opts=abc, q=2 -> 'c', x mới=3
  vị trí 2: block=  4, x=  3, opts=ab, q=0 -> 'a', x mới=3
  vị trí 3: block=  2, x=  3, opts=bc, q=1 -> 'c', x mới=1
  vị trí 4: block=  1, x=  1, opts=ab, q=1 -> 'b', x mới=0
  kết quả: cacb


'cacb'

Trace cách 2 (backtracking) để thấy thứ tự lá được đếm và điểm dừng:

In [18]:
def trace_backtrack(n, k):
    path, count = [], 0
    def dfs(depth):
        nonlocal count
        if len(path) == n:
            count += 1
            print("  " * depth + f"lá #{count}: {''.join(path)}" + ("  <-- DỪNG" if count == k else ""))
            return count == k
        for ch in "abc":
            if path and path[-1] == ch:
                continue
            path.append(ch)
            print("  " * depth + f"thử '{ch}' -> path={''.join(path)}")
            if dfs(depth + 1):
                return True
            path.pop()
        return False
    dfs(0)

trace_backtrack(2, 4)

thử 'a' -> path=a
  thử 'b' -> path=ab
    lá #1: ab
  thử 'c' -> path=ac
    lá #2: ac
thử 'b' -> path=b
  thử 'a' -> path=ba
    lá #3: ba
  thử 'c' -> path=bc
    lá #4: bc  <-- DỪNG


## 9. Code nộp LeetCode

Dùng cách 4 (đếm trực tiếp) — `O(n)`, ngắn gọn:

In [19]:
class Solution:
    def getHappyString(self, n: int, k: int) -> str:
        block = 1 << (n - 1)                 # số chuỗi bắt đầu bằng mỗi chữ
        if k > 3 * block:
            return ""
        q, x = divmod(k - 1, block)
        res = ["abc"[q]]
        for _ in range(n - 1):
            block //= 2
            opts = [ch for ch in "abc" if ch != res[-1]]
            q, x = divmod(x, block)
            res.append(opts[q])
        return "".join(res)

sol = Solution()
print([sol.getHappyString(n, k) for (n, k), _ in examples])

['c', '', 'cab']


## 10. Vì sao thuật toán đúng

- **Đếm:** chuỗi happy độ dài `m` sau khi cố định tiền tố khác rỗng: mỗi vị trí kế tiếp có đúng 2 lựa chọn (khác ký tự ngay trước) và lựa chọn nào cũng hợp lệ cho bước sau ⇒ số cách hoàn thành `r` vị trí còn lại là `2^r`, không phụ thuộc tiền tố.
- **Thứ tự:** trong danh sách đã sắp, mọi chuỗi có tiền tố `P + u` đứng trước mọi chuỗi có tiền tố `P + v` khi `u < v` (cùng độ dài, khác nhau lần đầu ở vị trí đó). Vì vậy các chuỗi cùng tiền tố tạo thành một **khối liên tiếp**, các khối xếp theo thứ tự chữ.
- **Bất biến của cách 4:** trước mỗi bước, `x` là chỉ số (từ 0) của đáp án **bên trong khối** ứng với tiền tố `res` hiện tại. Khối này chia thành các khối con kích thước `block` bằng nhau theo thứ tự `opts`; đáp án nằm trong khối con thứ `x // block`, ở vị trí `x % block` ⇒ bất biến được giữ. Sau `n` ký tự, khối chỉ còn 1 chuỗi = đáp án.
- **Cách 2 / 3:** duyệt cây theo thứ tự con tăng dần (DFS tiền thứ tự hoặc BFS từng tầng) liệt kê lá theo thứ tự từ điển, nên phần tử thứ `k` được đếm chính là đáp án.

## 11. Phân tích Time / Space

Gọi `n` là độ dài, `T = 3 · 2^(n-1)` là số happy string.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force `product` + lọc + sort | `O(3^n · n)` | `O(3^n · n)` | `n=10` ⇒ 59 049 chuỗi |
| 2. Backtracking dừng sớm | `O(min(k, T) · n)` | `O(n)` (đệ quy) | `k ≤ 100` ⇒ rất nhanh |
| 3. BFS hàng đợi | `O(T · n)` | `O(T · n)` | sinh hết tầng cuối |
| 4. Đếm trực tiếp | `O(n)` | `O(n)` | không phụ thuộc `k` |

Với ràng buộc nhỏ (`n ≤ 10`, `k ≤ 100`) mọi cách đều qua, nhưng cách 4 vẫn đúng khi `n` lên hàng nghìn và `k` cực lớn (Python có số nguyên lớn).

In [20]:
# Cách 4 vẫn chạy tức thì với n rất lớn (k là số nguyên lớn)
big_n = 60
big_k = 3 * 2 ** (big_n - 1)          # chuỗi cuối cùng
s = Solution().getHappyString(big_n, big_k)
print(len(s), s[:20], "...", is_happy(s))

60 cbcbcbcbcbcbcbcbcbcb ... True


## 12. Bộ test

1. Ví dụ của đề.
2. Edge case: `n = 1` (mọi `k` từ 1 đến 4), `k = T` (chuỗi cuối), `k = T + 1` (rỗng), `n = 10, k = 100`.
3. Đối chiếu **toàn bộ** cặp `(n, k)` với `n ∈ [1, 8]`, `k ∈ [1, min(100, T+2)]` giữa 5 cài đặt.
4. Random có seed cố định trên đúng ràng buộc đề.

In [21]:
impls = {
    "brute": happy_brute,
    "backtrack": happy_backtrack,
    "bfs": happy_bfs,
    "direct": happy_direct,
    "bits": happy_bits,
    "Solution": lambda n, k: Solution().getHappyString(n, k),
    "SolBT": lambda n, k: SolutionBacktrack().getHappyString(n, k),
    "SolBFS": lambda n, k: SolutionBFS().getHappyString(n, k),
}
# 1. ví dụ đề
for name, f in impls.items():
    for (n, k), want in examples:
        assert f(n, k) == want, (name, n, k)
# 2. edge
assert [Solution().getHappyString(1, k) for k in range(1, 5)] == ["a", "b", "c", ""]
assert Solution().getHappyString(2, 6) == "cb" and Solution().getHappyString(2, 7) == ""
assert Solution().getHappyString(10, 100) == happy_brute(10, 100)
assert Solution().getHappyString(3, 12) == "cbc" and Solution().getHappyString(3, 13) == ""
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [22]:
# 3. đối chiếu vét cạn với n nhỏ
cnt = 0
for n in range(1, 9):
    T = 3 * 2 ** (n - 1)
    ref_list = happy_list(n)
    assert len(ref_list) == T
    for k in range(1, min(100, T + 2) + 1):
        want = ref_list[k - 1] if k <= T else ""
        for name, f in impls.items():
            assert f(n, k) == want, (name, n, k)
        cnt += 1
print("đã đối chiếu", cnt, "cặp (n, k) x", len(impls), "cài đặt: OK")

đã đối chiếu 401 cặp (n, k) x 8 cài đặt: OK


In [23]:
# 4. random theo đúng ràng buộc (1<=n<=10, 1<=k<=100), seed cố định
rng = random.Random(1415)
for _ in range(300):
    n, k = rng.randint(1, 10), rng.randint(1, 100)
    want = happy_brute(n, k)
    got = Solution().getHappyString(n, k)
    assert got == want, (n, k, got, want)
    if got:
        assert len(got) == n and is_happy(got)
print("random 300 test: OK")

random 300 test: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `k` đếm từ 1 (dùng `k` làm chỉ số 0)

In [24]:
def wrong_index(n, k):
    lst = happy_list(n)
    return lst[k] if k < len(lst) else ""      # SAI: phải là lst[k-1]

print("n=3,k=9 ->", repr(wrong_index(3, 9)), "(đúng là 'cab')")
print("n=1,k=3 ->", repr(wrong_index(1, 3)), "(đúng là 'c')")

n=3,k=9 -> 'cac' (đúng là 'cab')
n=1,k=3 -> '' (đúng là 'c')


**Vì sao sai:** đề nói "chuỗi thứ `k`" theo kiểu con người đếm 1, 2, 3… Danh sách Python đánh chỉ số từ 0, nên chuỗi thứ `k` nằm ở `lst[k-1]`. Lỗi này còn làm `k = T` trả về `""` thay vì chuỗi cuối.

**Sửa:** dùng `lst[k - 1]` và điều kiện `k <= len(lst)`; trong cách đếm thì đặt `x = k - 1` trước khi chia.

In [25]:
def fixed_index(n, k):
    lst = happy_list(n)
    return lst[k - 1] if k <= len(lst) else ""

print(repr(fixed_index(3, 9)), repr(fixed_index(1, 3)))

'cab' 'c'


### Lỗi 2 — Đếm sai tổng số chuỗi (dùng `3^n` hoặc `2^n`)

In [26]:
def wrong_total(n, k):
    if k > 3 ** n:                              # SAI: tổng là 3*2^(n-1)
        return ""
    block = 1 << (n - 1)
    q, x = divmod(k - 1, block)
    if q >= 3:
        return "<<chỉ số ngoài bảng chữ>>"      # tránh văng IndexError để in ra được
    res = ["abc"[q]]
    for _ in range(n - 1):
        block //= 2
        opts = [ch for ch in "abc" if ch != res[-1]]
        q, x = divmod(x, block)
        res.append(opts[q])
    return "".join(res)

print("n=2,k=7 ->", repr(wrong_total(2, 7)), "(đúng là '' vì chỉ có 6 chuỗi)")

n=2,k=7 -> '<<chỉ số ngoài bảng chữ>>' (đúng là '' vì chỉ có 6 chuỗi)


**Vì sao sai:** `3^n` là số chuỗi **bất kỳ** trên `abc`, không phải số happy string. Với `n = 2, k = 7` có 9 > 7 nên không trả `""`, chương trình đi tiếp và tính ra chỉ số chữ đầu = 3 — ngoài bảng chữ (code thật sẽ ném `IndexError`).

**Sửa:** `total = 3 * 2 ** (n - 1)`; kiểm tra `k > total` trước.

In [27]:
print("n=2,k=7 ->", repr(Solution().getHappyString(2, 7)))

n=2,k=7 -> ''


### Lỗi 3 — Backtracking không dừng sớm đúng cách (chỉ `return` ở tầng lá)

In [28]:
def wrong_stop(n, k):
    path, found = [], []
    count = 0
    def dfs():
        nonlocal count
        if len(path) == n:
            count += 1
            if count == k:
                found.append("".join(path))
            return                              # chỉ thoát tầng lá
        for ch in "abc":
            if path and path[-1] == ch:
                continue
            path.append(ch)
            dfs()
            path.pop()
    dfs()
    return found[-1] if found else "", count

ans, visited = wrong_stop(10, 1)
print("kết quả", ans, "| số lá đã duyệt:", visited, "(chỉ cần 1!)")

kết quả ababababab | số lá đã duyệt: 1536 (chỉ cần 1!)


**Vì sao sai (về hiệu năng):** kết quả vẫn đúng, nhưng thuật toán duyệt **toàn bộ** 1536 lá dù chỉ cần lá đầu tiên. Ở bài lớn hơn (bảng chữ to, `n` lớn) điều này biến `O(k·n)` thành `O(T·n)` — dễ TLE.

**Sửa:** cho `dfs` trả về `True` khi đã tìm thấy và mọi tầng kiểm tra `if dfs(): return True` (như `happy_backtrack`).

In [29]:
calls = 0
def counted_backtrack(n, k):
    global calls
    path, count = [], 0
    def dfs():
        global calls
        nonlocal count
        calls += 1
        if len(path) == n:
            count += 1
            return count == k
        for ch in "abc":
            if path and path[-1] == ch:
                continue
            path.append(ch)
            if dfs():
                return True
            path.pop()
        return False
    dfs()
    return "".join(path)

print(counted_backtrack(10, 1), "| số lần gọi dfs:", calls)

ababababab | số lần gọi dfs: 11


### Lỗi 4 — So sánh với ký tự cuối khi `path` còn rỗng / chọn sai tập `opts`

In [30]:
def wrong_opts(n, k):
    block = 1 << (n - 1)
    if k > 3 * block:
        return ""
    q, x = divmod(k - 1, block)
    res = ["abc"[q]]
    for _ in range(n - 1):
        block //= 2
        opts = [ch for ch in "cba" if ch != res[-1]]    # SAI: thứ tự giảm dần
        q, x = divmod(x, block)
        res.append(opts[q])
    return "".join(res)

print("n=3,k=9 ->", wrong_opts(3, 9), "(đúng là 'cab')")
print("n=3,k=1 ->", wrong_opts(3, 1), "(đúng là 'aba')")

n=3,k=9 -> cbc (đúng là 'cab')
n=3,k=1 -> acb (đúng là 'aba')


**Vì sao sai:** khối con thứ 0 phải ứng với chữ **nhỏ hơn** trong hai chữ còn lại. Duyệt `"cba"` đảo ngược thứ tự khối con nên thứ tự kết quả không còn là từ điển. Tương tự, trong backtracking nếu viết `if path[-1] == ch` mà không kiểm tra `path` rỗng sẽ gây `IndexError` ở bước đầu.

**Sửa:** luôn duyệt `"abc"` theo thứ tự tăng, và viết `if path and path[-1] == ch: continue`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**chuỗi/hoán vị thứ k theo thứ tự từ điển**" ⇒ nghĩ ngay đến **đếm kích thước khối** rồi chia `divmod` (bài 60 Permutation Sequence, 440 K-th Smallest in Lexicographical Order, 1643 Kth Smallest Instructions).
- Ràng buộc nhỏ (`n ≤ 10`, `k ≤ 100`) ⇒ backtracking sinh theo thứ tự là đủ, viết nhanh, ít rủi ro trong phỏng vấn.
- **Sinh theo thứ tự = duyệt lựa chọn theo thứ tự tăng** — không cần sort.
- Khi số cách hoàn thành **không phụ thuộc** vào tiền tố (như ở đây: luôn `2^r`) ⇒ phép đếm rất đơn giản; nếu phụ thuộc thì cần DP đếm (`cnt[pos][last]`).
- Kiểm tra nhanh trước khi nộp: `(1,3)→"c"`, `(1,4)→""`, `(3,9)→"cab"`, `(3,12)→"cbc"`, `(3,13)→""`.

## 15. Biến thể của bài

1. **Bảng chữ `m` ký tự** (thay vì 3): tổng `m · (m-1)^(n-1)`, khối con kích thước `(m-1)^r`.
2. **Bài ngược:** cho một happy string, tìm thứ hạng `k` của nó (dùng cùng kích thước khối, cộng dồn).
3. **Đếm happy string** có thêm điều kiện (ví dụ không chứa `"abc"`) ⇒ cần DP đếm theo trạng thái.
4. **LeetCode 1641 Count Sorted Vowel Strings**, **1220 Count Vowels Permutation**: đếm chuỗi với luật chuyển giữa các ký tự.
5. **LeetCode 60 Permutation Sequence**: hoán vị thứ `k` — khối kích thước `(n-1)!`.

Code biến thể 1 và 2:

In [31]:
def kth_happy_m(n, k, m):
    alpha = "abcdefghijklmnopqrstuvwxyz"[:m]
    block = (m - 1) ** (n - 1)
    if k > m * block:
        return ""
    q, x = divmod(k - 1, block)
    res = [alpha[q]]
    for _ in range(n - 1):
        block //= (m - 1)
        opts = [ch for ch in alpha if ch != res[-1]]
        q, x = divmod(x, block)
        res.append(opts[q])
    return "".join(res)

def rank_of(s):
    n = len(s)
    block = 1 << (n - 1)
    x = "abc".index(s[0]) * block
    for i in range(1, n):
        block //= 2
        opts = [ch for ch in "abc" if ch != s[i - 1]]
        x += opts.index(s[i]) * block
    return x + 1

print(kth_happy_m(3, 9, 3), kth_happy_m(2, 5, 4), rank_of("cab"), rank_of("cbc"))

cab bc 9 12


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | `product` mọi chuỗi, lọc happy, sort, lấy `k-1` | `O(3^n·n)` | `O(3^n·n)` | Kiểm chứng, `n` rất nhỏ |
| 2. Backtracking dừng sớm | DFS chọn chữ khác chữ cuối theo `a<b<c`, đếm lá | `O(k·n)` | `O(n)` | Phỏng vấn, dễ viết, `k` nhỏ |
| 3. BFS hàng đợi | Mở rộng từng tầng, tầng cuối đã sắp | `O(2^n·n)` | `O(2^n·n)` | Khi cần cả danh sách |
| 4. Đếm trực tiếp / bit | Khối kích thước `2^r`, `divmod` từng vị trí | `O(n)` | `O(n)` | Tối ưu, `k` hoặc `n` lớn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().getHappyString(2, 4)` trả về gì? Còn `getHappyString(4, 25)`?

### Bài 2 (tính tay)
Dùng cách đếm, tính tay `getHappyString(4, 20)` (ghi `block`, `x`, `opts` từng bước) và `getHappyString(5, 1)`.

### Bài 3 (điền TODO)
Hoàn thiện `count_happy(n)` trả về số happy string độ dài `n` **bằng DP** `dp[last]` (không dùng công thức).

### Bài 4 (tự viết lại)
Viết `rank_happy(s)`: cho happy string `s`, trả về `k` (1-indexed) sao cho `getHappyString(len(s), k) == s`.

### Bài 5 (mini challenge)
Viết `kth_happy_alpha(n, k, alpha)`: bảng chữ là chuỗi `alpha` **chưa sắp xếp**, có thể có ký tự trùng (cần loại trùng). Trả chuỗi thứ `k` theo thứ tự từ điển hoặc `""`.

In [32]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_happy(n):
    dp = {"a": 1, "b": 1, "c": 1}       # số chuỗi độ dài 1 kết thúc bằng mỗi chữ
    for _ in range(n - 1):
        # TODO: new[ch] = tổng dp[p] với p != ch
        pass
    # TODO: trả về tổng dp
    pass

# Bài 4 — TODO
def rank_happy(s):
    # TODO: cộng dồn (vị trí chữ trong opts) * block cho từng vị trí
    pass

# Bài 5 — TODO
def kth_happy_alpha(n, k, alpha):
    # TODO: alpha = sorted(set(alpha)); m = len(alpha); xử lý m == 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê `n = 2`: `ab, ac, ba, bc, ca, cb`. (2) Chuỗi thứ 4 là `bc`. (3) `n = 4` có `3·8 = 24` chuỗi < 25 ⇒ `""`.

**Bài 2.** (1) `block = 8`, `x = 19` ⇒ `q = 2` → `c`, `x = 3`. (2) `block = 4`, opts `ab`, `q = 0` → `a`, `x = 3`; `block = 2`, opts `bc`, `q = 1` → `c`, `x = 1`. (3) `block = 1`, opts `ab`, `q = 1` → `b` ⇒ `"cacb"`. Chuỗi đầu tiên độ dài 5 luôn là `"ababa"`.

**Bài 3.** (1) `new = {ch: sum(v for p, v in dp.items() if p != ch) for ch in "abc"}`. (2) Gán `dp = new`. (3) Trả `sum(dp.values())`, kiểm tra bằng `3 * 2**(n-1)`.

**Bài 4.** (1) `block = 2^(n-1)`, chữ đầu đóng góp `"abc".index(s[0]) * block`. (2) Mỗi vị trí sau: `block //= 2`, cộng `opts.index(s[i]) * block`. (3) Cộng 1 ở cuối.

**Bài 5.** (1) `alpha = sorted(set(alpha))`, `m = len(alpha)`. (2) Nếu `m == 1`: chỉ có chuỗi khi `n == 1` (và `k == 1`). (3) Ngược lại dùng `block = (m-1)^(n-1)` như `kth_happy_m`.

## 19. Đáp án tham khảo

In [33]:
# Bài 1
assert Solution().getHappyString(2, 4) == "bc"
assert Solution().getHappyString(4, 25) == ""

# Bài 2
assert Solution().getHappyString(4, 20) == "cacb" == happy_brute(4, 20)
assert Solution().getHappyString(5, 1) == "ababa"

# Bài 3
def count_happy_sol(n):
    dp = {"a": 1, "b": 1, "c": 1}
    for _ in range(n - 1):
        dp = {ch: sum(v for p, v in dp.items() if p != ch) for ch in "abc"}
    return sum(dp.values())

for n in range(1, 15):
    assert count_happy_sol(n) == 3 * 2 ** (n - 1)

# Bài 4
def rank_happy_sol(s):
    n = len(s)
    block = 1 << (n - 1)
    x = "abc".index(s[0]) * block
    for i in range(1, n):
        block //= 2
        opts = [ch for ch in "abc" if ch != s[i - 1]]
        x += opts.index(s[i]) * block
    return x + 1

for n in range(1, 8):
    for k, s in enumerate(happy_list(n), start=1):
        assert rank_happy_sol(s) == k
        assert Solution().getHappyString(n, rank_happy_sol(s)) == s

# Bài 5
def kth_happy_alpha_sol(n, k, alpha):
    alpha = sorted(set(alpha))
    m = len(alpha)
    if m == 1:
        return alpha[0] if n == 1 and k == 1 else ""
    block = (m - 1) ** (n - 1)
    if k > m * block:
        return ""
    q, x = divmod(k - 1, block)
    res = [alpha[q]]
    for _ in range(n - 1):
        block //= (m - 1)
        opts = [ch for ch in alpha if ch != res[-1]]
        q, x = divmod(x, block)
        res.append(opts[q])
    return "".join(res)

def brute_alpha(n, k, alpha):
    a = sorted(set(alpha))
    lst = sorted("".join(t) for t in product(a, repeat=n) if is_happy("".join(t)))
    return lst[k - 1] if k <= len(lst) else ""

assert kth_happy_alpha_sol(3, 9, "cba") == "cab"
rng = random.Random(7)
for _ in range(200):
    alpha = "".join(rng.choice("xyzw") for _ in range(rng.randint(1, 5)))
    n, k = rng.randint(1, 5), rng.randint(1, 60)
    assert kth_happy_alpha_sol(n, k, alpha) == brute_alpha(n, k, alpha), (n, k, alpha)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phần tử thứ k theo thứ tự từ điển" bằng đếm khối:**

```python
def kth(n, k):
    total = so_cach(prefix_rong, n)
    if k > total: return ""
    x, res = k - 1, []
    for pos in range(n):
        for ch in cac_lua_chon_hop_le_tang_dan(res):
            size = so_cach_hoan_thanh(res + [ch])   # ở bài này = 2^(n-1-pos)
            if x < size:
                res.append(ch); break
            x -= size
    return "".join(res)
```

**Template backtracking sinh theo thứ tự + dừng sớm:**

```python
def dfs():
    if done(path): count += 1; return count == k
    for ch in sorted_choices:
        if not valid(path, ch): continue
        path.append(ch)
        if dfs(): return True
        path.pop()
    return False
```

**Checklist:**
- [ ] Đổi `k` (1-indexed) thành chỉ số 0 trước khi chia / truy cập list.
- [ ] Tổng số happy string là `3 · 2^(n-1)`, không phải `3^n`.
- [ ] Trả `""` khi `k` vượt tổng.
- [ ] Duyệt chữ theo thứ tự tăng `a, b, c` (không sort lại).
- [ ] Không so `path[-1]` khi `path` rỗng.
- [ ] Đã thử `(1,3)`, `(1,4)`, `(3,9)`, `(3,12)`, `(10,100)`.